# What Exploration Can Ignore: Duck fork, K playing

This notebook contains the code of our eligible entry that carries the agent (ref. 57852519): the Tufa Labs Duck harness (MIT) with graph explorer K in mode 1 (playing).

This is a presentation-edited copy of version 2 of the entry kernel. Its executable code changes only comments, docstrings, log messages, identifier names, and internal string labels (configuration values and counter keys translated into English). Abstract-syntax-tree comparison after those normalizations finds the code identical.

- **Writeup**: [link]
- **Paper (PDF)**: [link]

### Attribution & Licenses
- **Tufa Labs Duck Harness**: based on `jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner` (MIT License), by Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit, Michal Tesnar, and Stefano Viel.
- **TAAF**: Tufa ARC-AGI Framework and ARC3-Inference.
- **Model weights & wheelhouse**: `driessmit1/arc3-vllm-h100-wheelhouse-v3`, `jeroencottaar/taaf-kaggle-source-share`, and `driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot`.

### Reproduction
- **Competition rerun (`KAGGLE_IS_COMPETITION_RERUN=1`)**: connects to the competition gateway and evaluates on the live competition Arcade.
- **Interactive execution / phase A (`KAGGLE_IS_COMPETITION_RERUN=0`)**: evaluates the fixed public games `vc33`, `tu93`, and `ls20` offline, without gateway connectivity.
- **Interactive setup:** select an RTX Pro 6000 GPU and use **Save & Run All**. Internet is disabled.


## What this notebook contains

This entry scored **0.75** on the leaderboard. Our other eligible entry, the intact dfranzen base, scored **26.74** (ref. 57881859). The two bases differ and no paired control of the Duck base was scored, so the gap is not an estimate of K's effect (paper, Section 5).

- **Graph exploration:** K hashes frames into states with display-band cells masked, records tested actions and transitions, and plans breadth-first to the nearest state with untested actions.
- **Playing mode (mode 1):** the language model proposes actions by default, and K observes every transition, prunes certified keys from the model's action list, and takes the turn on named triggers such as model failure, stall, repeated no-op, or time pressure. K's graph notes enter the model's prompt.
- **No-effect certificates (H8):** actions observed to change nothing outside the display band can be certified and masked from the model's action list. A later contradiction falsifies the certificate, which revokes it and voids the level's other certificates. In a hidden-state level, no-op statements in the model's prompt are downgraded and no longer presented as proof.
- **Scope:** this notebook contains the earlier explorer and certificate code. Win-condition identification (H5), the later dfranzen-base versions (advising and last-resort modes), the cross-game memory, and the separate 25-game local bench are not included.

The notebook installs the ARC runtime offline, loads the bundled benchmark, runs the configured games, and writes results and diagnostics to `/kaggle/working`.


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimizes diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["jeroencottaar/taaf-kaggle-source-share", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), and then run
the solver's setup commands (installing wheels, fetching model weights, and so on).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 6. Graph explorer and hybrid analyzer

Write `graph_explorer.py` and `hybrid_analyzer.py`, and then install the eligible mode-1 analyzer hook. The hybrid layer observes model actions, maintains K's graph and certificates, adds K's graph notes to the prompt, and prunes certified keys. On the active triggers, K takes the turn instead of the model.


In [ ]:
%%writefile /kaggle/working/graph_explorer.py
"""Graph Explorer K: frontier-driven BFS graph exploration with goal hypotheses and life budgeting for ARC-AGI-3.

Architecture and exploration strategy:
- State abstraction: node key = frame pixels (with HUD masked) + completed levels.
  Each graph node stores raw frame bytes, candidate actions categorized by priority tier,
  tried actions, and directed action->node transition edges.
- HUD mask discovery: static/timer regions detected across transitions are masked; when the mask
  updates, the node index is reindexed from raw frames, merging equivalent states.
- Priority tiers: tier 0 = available directional keys; tier 1 = non-background connected component
  centers (sorted by size); tier 2 = secondary component centers; tier 3 = ACTION7; tier 4 = random grid clicks.
- Planning & Decision: if current node has an untried candidate at priority <= p, sample it;
  otherwise, BFS along known edges to the nearest node with untried actions; if none, increment p.
- Life budget & hidden-state handling: repeated deaths at consistent step counts infer life timers,
  triggering voluntary reset before lethal steps. Deaths on previously safe edges are treated as
  hidden-state events rather than deterministic lethal edges.
- Goal hypotheses (H1 & H2):
  * Directional motion (H1): tracks mobile color components and static target components to bias
    action selection towards goal alignment.
  * Work area vs reference window (H2): identifies dynamic work regions and static reference windows,
    biasing BFS and action selection towards minimizing cell discrepancy.
"""
from __future__ import annotations

import random
import time
import zlib
from collections import Counter, deque
from typing import Any

from arcengine import FrameData, GameAction, GameState

from agents.agent import Agent

NOOP, CHANGE, PROGRESS, DEATH = 0, 1, 2, 3
TIER_KEYS, TIER_CLICK_A, TIER_CLICK_B, TIER_UNDO, TIER_RANDOM = 0, 1, 2, 3, 4
MAX_TIER = TIER_RANDOM
MAX_COMPS = 128            # candidate components per state
RANDOM_CLICKS = 12
REOPEN_MAX = 8             # max times tested actions may be reopened per level
BFS_LIMIT = 20000
EXHAUST_RESET = 2          # exhausted: alternates between reopening tested actions and level reset
HUD_BORDER = 4
HUD_MIN_EVENTS = 3
HUD_MAX_STEP = 3
HUD_CELL_MIN = 5
HUD_RATE = 0.9
TIME_LIMIT_S = 20 * 60
TARGET_PATIENCE = 15       # actions near target without progress before switching target
FRONTIER_SLACK = 3         # extra steps beyond nearest frontier accepted by BFS for higher-priority target
DIR_MIN = 2                # observations required to trust directional displacement of a key
H2_MIN_OVERLAP = 0.5       # fraction of work area structure that reference area must cover
H2_MIN_CHANGED = 4         # changed cells threshold before searching for work/reference areas
H2_PATIENCE = 150          # guided actions without reducing discrepancy falsifies hypothesis and disables guidance
TIMER_HITS = 3             # deaths at consistent step counts required to infer life timer budget


class Node:
    __slots__ = ("key", "level", "raw", "shape", "cands", "tier", "cls", "tried", "edges", "visits", "mover", "legal")

    def __init__(self, key: int, level: int, raw: bytes, shape: tuple[int, int]) -> None:
        self.key = key
        self.level = level
        self.raw = raw
        self.shape = shape
        self.cands: list[tuple] = []
        self.tier: dict[tuple, int] = {}
        self.cls: dict[tuple, tuple] = {}            # click candidate -> component class (color, size, width, height)
        self.tried: dict[tuple, int] = {}
        self.edges: dict[tuple, int] = {}
        self.visits = 0
        self.mover: tuple[float, float] | None = None
        self.legal: frozenset[int] = frozenset()      # available action ids on last visit (can vary without changing pixels)

    def eff_tier(self, c: tuple, reactive: set[tuple]) -> int:
        k = self.cls.get(c)
        if k is not None and k in reactive:
            return TIER_CLICK_A
        return self.tier[c]

    def untried(self, p: int, reactive: set[tuple]) -> list[tuple]:
        return [c for c in self.cands if c not in self.tried and (not self.legal or c[0] in self.legal)
                and self.eff_tier(c, reactive) <= p]

    def merge(self, other: "Node") -> None:
        for c in other.cands:
            if c not in self.tier:
                self.cands.append(c)
                self.tier[c] = other.tier[c]
                if c in other.cls:
                    self.cls[c] = other.cls[c]
        self.tried.update(other.tried)
        self.edges.update(other.edges)
        self.visits += other.visits
        self.legal = self.legal | other.legal          # reindexing merges visits across differing availability sets


class MyAgent(Agent):
    MAX_ACTIONS = 80

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.rng = random.Random(zlib.crc32(self.game_id.encode()))
        self.t0 = time.time()
        self.nodes: dict[int, Node] = {}
        self.p_level: dict[int, int] = {}
        self.exhaust: dict[int, int] = {}
        self.reopened: dict[int, int] = {}
        self.reactive: set[tuple] = set()           # component classes that already responded to click actions
        self.cls_tries: Counter[tuple] = Counter()
        self.plan: deque[tuple[tuple, int]] = deque()
        self.prev_grid: list[list[int]] | None = None
        self.prev_key: int | None = None
        self.prev_act: tuple | None = None
        self.prev_levels = 0
        self._click: dict[str, int] | None = None   # instance click coordinates avoiding shared mutable state on Enum
        self.steps = 0
        self.ui: set[tuple[int, int]] = set()
        self.hud: dict[int, dict] = {}
        self.hud_reset = False
        self.last_effective: set[tuple[int, int]] = set()
        self.stats = Counter()
        self.life = 0                                  # action count since last reset
        self.death_at: Counter[int] = Counter()
        self.max_life = 0
        self.life_budget: int | None = None
        self.start_key: dict[int, int] = {}            # level -> root node (life 0)
        self.dir_map: dict[int, Counter] = {a: Counter() for a in (1, 2, 3, 4)}
        self.changed_ever: dict[int, set[tuple[int, int]]] = {}
        self.target_idx: dict[int, int] = {}
        self.target_wait: dict[int, int] = {}
        self.tcache: dict[tuple, list] = {}
        self.guide_off: set[int] = set()               # levels where graph was exhausted; hypothesis failed, resume exploration
        self.h2cache: dict[tuple, tuple | None] = {}    # (level, changed_count) -> (x0, y0, bw, bh, rx, ry) or None
        self.h2_best: dict[int, int] = {}               # level -> minimum observed cell discrepancy
        self.h2_stale: dict[int, int] = {}              # level -> count of guided actions without improvement

    @property
    def name(self) -> str:
        return f"{super().name}.{self.MAX_ACTIONS}"

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        if latest_frame.state is GameState.WIN:
            return True
        return (time.time() - self.t0) > TIME_LIMIT_S

    # ---------- frame processing ----------
    @staticmethod
    def _grid(frame: FrameData) -> list[list[int]]:
        return frame.frame[-1] if frame.frame else []

    # ---------- HUD mask discovery ----------
    @staticmethod
    def _line(x: int, y: int, h: int, w: int):
        if y < HUD_BORDER or y >= h - HUD_BORDER:
            return ("r", y)
        if x < HUD_BORDER or x >= w - HUD_BORDER:
            return ("c", x)
        return None

    def _hud_learn(self, level: int, changed_vals: list[tuple[int, int, int, int]], h: int, w: int) -> None:
        hud = self.hud.setdefault(level, {"_n": 0, "_cells": {}, "bars": {}})
        hud["_n"] += 1
        n = hud["_n"]
        if self.hud_reset:
            for e in hud["bars"].values():
                e["cells"] = set()
            self.hud_reset = False
        groups: dict[tuple, set] = {}
        for x, y, f, t in changed_vals:
            line = self._line(x, y, h, w)
            if line is not None:
                groups.setdefault((line, f, t), set()).add((x, y))
        bars = hud["bars"]
        mirror = {k for k in groups if (k[0], k[2], k[1]) in groups}
        gained: dict[int, int] = {}
        for (line, f, t), cells in groups.items():
            gained[t] = gained.get(t, 0) + len(cells)
        for (line, f, t), cells in groups.items():
            if (line, f, t) in mirror or gained.get(f, 0) > 0:
                continue
            for c in cells:
                rec = hud["_cells"].setdefault(c, [0, n])
                rec[0] += 1
        for (line, f, t), cells in groups.items():
            if (line, f, t) in mirror:
                bars.setdefault((line, f, t), {"cells": set(), "seen": set(), "events": 0, "bad": False})["bad"] = True
                continue
            rev = bars.get((line, t, f))
            if rev is not None and rev["events"] > 0 and (cells & rev["cells"] or len(cells) > HUD_MAX_STEP):
                if len(cells) > HUD_MAX_STEP:
                    rev["cells"] = set()
                else:
                    rev["bad"] = True
                continue
            e = bars.setdefault((line, f, t), {"cells": set(), "seen": set(), "events": 0, "bad": False})
            if e["bad"]:
                continue
            if cells & e["cells"]:
                e["bad"] = True
                continue
            if e["cells"]:
                pos = (lambda c: c[0]) if line[0] == "r" else (lambda c: c[1])
                if min(abs(pos(p) - pos(q)) for p in cells for q in e["cells"]) > HUD_MAX_STEP:
                    e["bad"] = True
                    continue
            if len(cells) > HUD_MAX_STEP and e["events"] < HUD_MIN_EVENTS:
                e["bad"] = True
                continue
            e["events"] += 1
            e["cells"] |= cells
            e["seen"] |= cells

    def _hud_keys(self, level: int):
        hud = self.hud.get(level)
        if not hud:
            return [], set()
        keys = [k for k, e in hud["bars"].items()
                if not e["bad"] and e["events"] >= HUD_MIN_EVENTS and len(e["seen"]) >= HUD_MIN_EVENTS]
        n = hud["_n"]
        counters = {c for c, (k, first) in hud["_cells"].items()
                    if k >= HUD_CELL_MIN and k / max(1, n - first + 1) >= HUD_RATE}
        return keys, counters

    def _hud_mask(self, level: int, grid: list[list[int]]) -> set[tuple[int, int]]:
        keys, counters = self._hud_keys(level)
        m = set(counters)
        h = len(grid)
        w = len(grid[0]) if h else 0
        bars = self.hud.get(level, {}).get("bars", {})
        for (line, f, t) in keys:
            axis, idx = line
            cells = [(x, idx) for x in range(w)] if axis == "r" else [(idx, y) for y in range(h)]
            ok = [grid[y][x] in (f, t) for x, y in cells]
            e = bars[(line, f, t)]
            seed = e["seen"] | e["cells"]
            i = 0
            while i < len(cells):
                if not ok[i]:
                    i += 1
                    continue
                j = i
                while j < len(cells) and ok[j]:
                    j += 1
                if any(cells[k] in seed for k in range(i, j)):
                    m.update(cells[i:j])
                i = j
        return m

    @staticmethod
    def _is_hud_change(keys, counters, x: int, y: int, vb: int, va: int) -> bool:
        if (x, y) in counters:
            return True
        for (axis, idx), f, t in keys:
            if (y if axis == "r" else x) == idx and vb in (f, t) and va in (f, t):
                return True
        return False

    def _update_mask(self, grid: list[list[int]], level: int) -> set[tuple[int, int]]:
        """Compares with previous grid to update HUD mask; returns coordinates of cells changed outside HUD."""
        vals: list[tuple[int, int, int, int]] = []
        pg = self.prev_grid
        h, w = len(grid), len(grid[0]) if grid else 0
        if pg is not None and len(pg) == len(grid):
            for y, (r0, r1) in enumerate(zip(pg, grid)):
                if r0 != r1:
                    for x, (a, b) in enumerate(zip(r0, r1)):
                        if a != b:
                            vals.append((x, y, a, b))
            self.steps += 1
            if level == self.prev_levels:
                self._hud_learn(level, vals, h, w)
        keys, counters = self._hud_keys(level)
        eff = {(x, y) for x, y, a, b in vals if not self._is_hud_change(keys, counters, x, y, a, b)}
        new_ui = self._hud_mask(level, grid) if grid else set()
        if new_ui != self.ui:
            self.ui = new_ui
            self._reindex()
        self.last_effective = eff
        return eff

    # ---------- graph representation ----------
    def _key_of(self, grid: list[list[int]], levels: int) -> int:
        if not self.ui:
            return hash((levels, tuple(tuple(r) for r in grid)))
        rows = []
        for y, r in enumerate(grid):
            rows.append(tuple(-1 if (x, y) in self.ui else v for x, v in enumerate(r)))
        return hash((levels, tuple(rows)))

    @staticmethod
    def _pack(grid: list[list[int]]) -> bytes:
        return bytes(v & 0xFF for r in grid for v in r)

    @staticmethod
    def _unpack(raw: bytes, shape: tuple[int, int]) -> list[list[int]]:
        h, w = shape
        return [list(raw[y * w:(y + 1) * w]) for y in range(h)]

    def _reindex(self) -> None:
        """Recomputes node keys using raw frames when HUD mask changes, merging newly equivalent nodes."""
        self.stats["reindex"] += 1
        old = list(self.nodes.values())
        remap: dict[int, int] = {}
        fresh: dict[int, Node] = {}
        for n in old:
            k = self._key_of(self._unpack(n.raw, n.shape), n.level)
            remap[n.key] = k
            if k in fresh:
                fresh[k].merge(n)
            else:
                n.key = k
                fresh[k] = n
        for n in fresh.values():
            n.edges = {a: remap.get(t, t) for a, t in n.edges.items()}
        self.nodes = fresh
        if self.prev_key is not None:
            self.prev_key = remap.get(self.prev_key, self.prev_key)
        self.plan.clear()

    def _click_targets(self, grid: list[list[int]]) -> list[tuple[int, int]]:
        h = len(grid)
        w = len(grid[0]) if h else 0
        if not h or not w:
            return []
        bg = Counter(c for row in grid for c in row).most_common(1)[0][0]
        seen = [[False] * w for _ in range(h)]
        comps: list[tuple[int, int, int]] = []
        for y in range(h):
            for x in range(w):
                if seen[y][x] or grid[y][x] == bg or (x, y) in self.ui:
                    continue
                col = grid[y][x]
                stack = [(x, y)]
                seen[y][x] = True
                cells = []
                while stack:
                    cx, cy = stack.pop()
                    cells.append((cx, cy))
                    for nx, ny in ((cx + 1, cy), (cx - 1, cy), (cx, cy + 1), (cx, cy - 1)):
                        if 0 <= nx < w and 0 <= ny < h and not seen[ny][nx] and grid[ny][nx] == col:
                            seen[ny][nx] = True
                            stack.append((nx, ny))
                mx = sum(c[0] for c in cells) // len(cells)
                my = sum(c[1] for c in cells) // len(cells)
                cx, cy = min(cells, key=lambda c: abs(c[0] - mx) + abs(c[1] - my))
                xs = [c[0] for c in cells]
                ys = [c[1] for c in cells]
                comps.append((len(cells), cx, cy, (int(col), len(cells), max(xs) - min(xs) + 1, max(ys) - min(ys) + 1)))
        comps.sort(key=lambda t: (t[0], t[3][0], t[1], t[2]))
        out = []
        for _, cx, cy, k in comps[:MAX_COMPS]:
            if w != 64 or h != 64:
                cx, cy = cx * 64 // w, cy * 64 // h
            out.append((cx, cy, k))
        return out

    def _node(self, key: int, grid: list[list[int]], levels: int, avail: list[int]) -> Node:
        n = self.nodes.get(key)
        if n is None:
            n = Node(key, levels, self._pack(grid), (len(grid), len(grid[0]) if grid else 0))
            self.nodes[key] = n
        # Node key is pixels + level; available_actions may change across visits without altering frame pixels. Each
        # visit reconciles: newly available keys become untried candidates, and legal set updates to current availability.
        self._seed_cands(n, grid, avail)
        n.legal = frozenset(avail)
        return n

    def _seed_cands(self, n: Node, grid: list[list[int]], avail: list[int]) -> None:
        for a in avail:
            if a in (1, 2, 3, 4, 5) and (a,) not in n.tier:
                n.cands.append((a,))
                n.tier[(a,)] = TIER_KEYS
        if 6 in avail and not any(c[0] == 6 for c in n.cands):
            seen_cls: set[tuple] = set()
            for x, y, k in self._click_targets(grid):
                c = (6, x, y)
                if c in n.tier:
                    continue
                n.cands.append(c)
                n.cls[c] = k
                n.tier[c] = TIER_CLICK_A if k not in seen_cls else TIER_CLICK_B
                seen_cls.add(k)
        if 7 in avail and (7,) not in n.tier:
            n.cands.append((7,))
            n.tier[(7,)] = TIER_UNDO

    def _add_random_clicks(self, n: Node) -> int:
        added = 0
        for _ in range(RANDOM_CLICKS):
            c = (6, self.rng.randrange(64), self.rng.randrange(64))
            if c not in n.tier:
                n.cands.append(c)
                n.tier[c] = TIER_RANDOM
                added += 1
        return added

    # ---------- directional goal hypothesis: mobile component, target, heading ----------
    @staticmethod
    def _centroid(cells) -> tuple[float, float]:
        n = len(cells)
        return (sum(c[0] for c in cells) / n, sum(c[1] for c in cells) / n)

    def _learn_mover(self, node: Node, pn: Node, act: tuple, pg: list[list[int]], grid: list[list[int]],
                     effective: set[tuple[int, int]], res: int) -> None:
        if res == PROGRESS:
            return
        if act[0] not in (1, 2, 3, 4) or res != CHANGE or not effective:
            if node.mover is None and pn.mover is not None and act[0] not in (1, 2, 3, 4):
                node.mover = pn.mover
            return
        # color pair (C->F, F->C) with matching counts indicates color C moved over background F
        trans: Counter = Counter((pg[y][x], grid[y][x]) for (x, y) in effective)
        pairs = [(c, f) for (c, f), n in trans.items() if c < f and trans.get((f, c), 0) == n]
        if len(pairs) != 1:                      # ambiguous (multiple moving entities) or no exact count match: skip displacement update
            return
        c, f = pairs[0]
        appeared = [(x, y) for (x, y) in effective if pg[y][x] == f and grid[y][x] == c]
        vanished = [(x, y) for (x, y) in effective if pg[y][x] == c and grid[y][x] == f]
        if not self._connected(appeared) and not self._connected(vanished):
            return
        if not self._connected(appeared):        # swap if c was the background
            c, f = f, c
            appeared, vanished = vanished, appeared
        cv = self._centroid(vanished)
        ca = self._centroid(appeared)
        dx, dy = ca[0] - cv[0], ca[1] - cv[1]
        sx = 0 if abs(dx) < 0.5 else (1 if dx > 0 else -1)
        sy = 0 if abs(dy) < 0.5 else (1 if dy > 0 else -1)
        if (sx, sy) != (0, 0):
            self.dir_map[act[0]][(sx, sy)] += 1
        node.mover = ca
        self.stats["mover_moves"] += 1

    @staticmethod
    def _connected(cells) -> bool:
        if not cells:
            return False
        s = set(cells)
        q = [cells[0]]
        seen = {cells[0]}
        while q:
            x, y = q.pop()
            for nb in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
                if nb in s and nb not in seen:
                    seen.add(nb)
                    q.append(nb)
        return len(seen) == len(s)

    def _dir_of(self, a: int) -> tuple[int, int] | None:
        c = self.dir_map.get(a)
        if not c:
            return None
        (d, n), = c.most_common(1)
        if n < DIR_MIN or n / sum(c.values()) < 0.6:
            return None
        return d

    def _targets(self, level: int, grid: list[list[int]]) -> list[tuple[float, float]]:
        ch = self.changed_ever.get(level, set())
        ck = (level, len(ch), len(self.ui))
        if ck in self.tcache:
            return self.tcache[ck]
        h = len(grid)
        w = len(grid[0]) if h else 0
        bg = Counter(v for r in grid for v in r).most_common(1)[0][0]
        seen = [[False] * w for _ in range(h)]
        comps = []
        for y in range(h):
            for x in range(w):
                if seen[y][x] or grid[y][x] == bg or (x, y) in self.ui:
                    continue
                col = grid[y][x]
                stack = [(x, y)]
                seen[y][x] = True
                cells = []
                static = True
                while stack:
                    cx, cy = stack.pop()
                    cells.append((cx, cy))
                    if (cx, cy) in ch:
                        static = False
                    for nx, ny in ((cx + 1, cy), (cx - 1, cy), (cx, cy + 1), (cx, cy - 1)):
                        if 0 <= nx < w and 0 <= ny < h and not seen[ny][nx] and grid[ny][nx] == col:
                            seen[ny][nx] = True
                            stack.append((nx, ny))
                if static and len(cells) <= 0.25 * h * w:
                    comps.append((int(col), len(cells), self._centroid(cells)))
        rar = Counter(c[0] for c in comps)
        comps.sort(key=lambda t: (rar[t[0]], t[1], t[2]))
        out = [c[2] for c in comps]
        self.tcache[ck] = out
        return out

    def _target(self, level: int, grid: list[list[int]]) -> tuple[float, float] | None:
        ts = self._targets(level, grid)
        if not ts:
            return None
        return ts[self.target_idx.get(level, 0) % len(ts)]

    @staticmethod
    def _l1(a: tuple[float, float], b: tuple[float, float]) -> float:
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _goal_score(self, c: tuple, mover, target) -> int:
        """Returns +1 if action reduces distance to target on axis, -1 if distance increases, 0 if unknown."""
        if mover is None or target is None or c[0] not in (1, 2, 3, 4):
            return 0
        d = self._dir_of(c[0])
        if d is None:
            return 0
        tx = 0 if abs(target[0] - mover[0]) < 0.5 else (1 if target[0] > mover[0] else -1)
        ty = 0 if abs(target[1] - mover[1]) < 0.5 else (1 if target[1] > mover[1] else -1)
        return d[0] * tx + d[1] * ty

    # ---------- discrepancy hypothesis: work region and reference window ----------
    def _h2_regions(self, level: int, grid: list[list[int]]):
        ch = self.changed_ever.get(level, set())
        ck = (level, len(ch))
        if ck in self.h2cache:
            return self.h2cache[ck]
        res = None
        h = len(grid)
        w = len(grid[0]) if h else 0
        cells = [c for c in ch if c not in self.ui]
        if len(cells) >= H2_MIN_CHANGED and h and w:
            bg = Counter(v for r in grid for v in r).most_common(1)[0][0]
            xs = [c[0] for c in cells]
            ys = [c[1] for c in cells]
            x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
            bw, bh = x1 - x0 + 1, y1 - y0 + 1
            if 2 <= bw and 2 <= bh and bw * bh <= 0.25 * h * w:
                wmask = [[grid[y0 + j][x0 + i] != bg for i in range(bw)] for j in range(bh)]
                wstruct = sum(sum(r) for r in wmask)
                blocked = ch | self.ui
                best = None
                for ry in range(0, h - bh + 1):
                    for rx in range(0, w - bw + 1):
                        if not (rx + bw <= x0 or rx >= x0 + bw or ry + bh <= y0 or ry >= y0 + bh):
                            continue
                        overlap = 0
                        cols: set[int] = set()
                        ok = True
                        for j in range(bh):
                            row = grid[ry + j]
                            for i in range(bw):
                                if (rx + i, ry + j) in blocked:
                                    ok = False
                                    break
                                v = row[rx + i]
                                if v != bg:
                                    cols.add(v)
                                    if wmask[j][i]:
                                        overlap += 1
                            if not ok:
                                break
                        if ok and len(cols) >= 2 and (best is None or overlap > best[0]):
                            best = (overlap, rx, ry)
                if best is not None and wstruct and best[0] >= H2_MIN_OVERLAP * wstruct:
                    res = (x0, y0, bw, bh, best[1], best[2])
                    if ck[1] and self.stats.get("h2_ref_level_%d" % level) is None:
                        self.stats["h2_ref_level_%d" % level] = 1
        self.h2cache[ck] = res
        return res

    @staticmethod
    def _h2_mismatch(grid: list[list[int]], reg) -> int:
        x0, y0, bw, bh, rx, ry = reg
        return sum(1 for j in range(bh) for i in range(bw) if grid[y0 + j][x0 + i] != grid[ry + j][rx + i])

    def _h2_mismatch_node(self, n: Node, reg) -> int:
        return self._h2_mismatch(self._unpack(n.raw, n.shape), reg)

    @staticmethod
    def _h2_cell_mismatch(grid: list[list[int]], reg, x: int, y: int) -> bool:
        x0, y0, bw, bh, rx, ry = reg
        if not (x0 <= x < x0 + bw and y0 <= y < y0 + bh):
            return False
        return grid[y][x] != grid[ry + (y - y0)][rx + (x - x0)]

    def _bfs(self, start: Node, p: int, target=None, reg=None) -> list[tuple[tuple, int]] | None:
        """Finds shortest path along known transitions to nearest node containing untried candidates at priority <= max_p."""
        parent: dict[int, tuple[int, tuple] | None] = {start.key: None}
        depth: dict[int, int] = {start.key: 0}
        q = deque([start.key])
        found: int | None = None
        best_d = None
        d_min = None
        while q and len(parent) < BFS_LIMIT:
            k = q.popleft()
            n = self.nodes.get(k)
            if n is None:
                continue
            if d_min is not None and depth[k] > d_min + FRONTIER_SLACK:
                break
            if k != start.key and n.untried(p, self.reactive):
                if d_min is None:
                    d_min = depth[k]
                    found = k
                    if target is not None and n.mover is not None:
                        best_d = self._l1(n.mover, target)
                    elif reg is not None:
                        best_d = self._h2_mismatch_node(n, reg)
                    else:
                        break
                else:
                    if target is not None and n.mover is not None:
                        dd = self._l1(n.mover, target)
                    elif target is None and reg is not None:
                        dd = self._h2_mismatch_node(n, reg)
                    else:
                        continue
                    if dd < best_d:
                        best_d, found = dd, k
                        self.stats["guided_frontier"] += 1
                continue
            for a, t in n.edges.items():
                if t == k or t in parent:
                    continue
                tn = self.nodes.get(t)
                if tn is None or tn.level != start.level:
                    continue
                parent[t] = (k, a)
                depth[t] = depth[k] + 1
                q.append(t)
        if found is None:
            return None
        path: list[tuple[tuple, int]] = []
        k = found
        while parent[k] is not None:
            pk, a = parent[k]
            path.append((a, k))
            k = pk
        path.reverse()
        return path

    # ---------- action selection & planning ----------
    def _reset(self, why: str) -> GameAction:
        self.prev_key = self.prev_act = None
        self.prev_grid = None
        self.plan.clear()
        self.hud_reset = True
        self.life = 0
        self.stats["reset_" + why] += 1
        act = GameAction.RESET
        act.reasoning = {"why": why}
        return act

    def _emit(self, n: Node, c: tuple, how: str) -> GameAction:
        if n.legal and c[0] not in n.legal:
            # Ensure proposed action is currently legal before mutating state; discard stale candidates or plans
            # inherited from visits with differing availability. Single legality check before state mutation.
            self.stats["illegal_action_avoided"] += 1
            if how == "plan":
                self.plan.clear()
            pool = n.untried(MAX_TIER, self.reactive) or [k for k in n.cands if k[0] in n.legal] \
                or [(a,) for a in sorted(n.legal)]
            c, how = self.rng.choice(pool), "legal"
        self.prev_key, self.prev_act = n.key, c
        self.life += 1
        self.max_life = max(self.max_life, self.life)
        self.stats[how] += 1
        act = GameAction.from_id(c[0])
        act.reasoning = {"how": how, "nodes": len(self.nodes), "p": self.p_level.get(n.level, 0)}
        # Action enum member is shared across threads; click coordinates are kept on instance
        # (_click, prev_act) to avoid cross-instance collision.
        self._click = {"x": int(c[1]), "y": int(c[2])} if act.is_complex() and len(c) == 3 else None
        return act

    def do_action_request(self, action: GameAction) -> FrameData:
        """Standalone harness entry point: submits action with instance-specific click coordinates."""
        if action.is_complex() and self._click:
            data = dict(self._click)
        else:
            data = action.action_type().model_dump()     # fresh instance: isolated state
        reasoning = getattr(action, "reasoning", None)
        if reasoning is not None and not isinstance(reasoning, dict):
            reasoning = {"text": str(reasoning)}
        raw = self.arc_env.step(action, data=data, reasoning=reasoning)
        return self._convert_raw_frame_data(raw)

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        levels = latest_frame.levels_completed
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            if latest_frame.state is GameState.GAME_OVER:
                self.death_at[self.life] += 1
                if self.life_budget is None and self.death_at[self.life] >= TIMER_HITS and self.life >= self.max_life:
                    self.life_budget = self.life
                    self.stats["timer"] = self.life
            if self.prev_key is not None and self.prev_act is not None and self.prev_key in self.nodes:
                pn = self.nodes[self.prev_key]
                safe_before = pn.tried.get(self.prev_act) in (NOOP, CHANGE, PROGRESS)
                timed = self.life_budget is not None and self.life >= self.life_budget
                if safe_before or timed:
                    self.stats["hidden_death"] += 1
                    if not safe_before:
                        pn.tried.pop(self.prev_act, None)
                else:
                    pn.tried[self.prev_act] = DEATH
                    pn.edges.pop(self.prev_act, None)
            return self._reset("game_over" if latest_frame.state is GameState.GAME_OVER else "start")

        grid = self._grid(latest_frame)
        if not grid:
            return self._reset("empty_frame")
        effective = self._update_mask(grid, levels)
        key = self._key_of(grid, levels)
        avail = [a for a in latest_frame.available_actions if a != 0]
        if not avail:                                  # When only RESET, or no actions, are available:
            return self._reset("only_reset_available")  # RESET is the only legal action:
        node = self._node(key, grid, levels, avail)
        node.visits += 1
        if levels > self.prev_levels:
            self.life = 0                              # new level: reset life budget tracker
            if self.plan:                              # clear pending plan on level transition to avoid cross-level action bleed
                self.stats["plan_new_level"] += 1
                self.plan.clear()
        if self.life == 0:
            self.start_key[levels] = key

        # 1. record directed edge from previous action
        if self.prev_key is not None and self.prev_act is not None and self.prev_key in self.nodes:
            pn = self.nodes[self.prev_key]
            if levels > self.prev_levels:
                res = PROGRESS
            elif effective or key != pn.key:
                res = CHANGE
            else:
                res = NOOP
            pn.tried[self.prev_act] = res
            pn.edges[self.prev_act] = key
            if self.prev_grid is not None and levels == self.prev_levels:
                self.changed_ever.setdefault(levels, set()).update(effective)
                self._learn_mover(node, pn, self.prev_act, self.prev_grid, grid, effective, res)
            k = pn.cls.get(self.prev_act)
            if k is not None:
                self.cls_tries[k] += 1
                if res != NOOP:
                    self.reactive.add(k)
            if self.plan:
                exp_act, exp_key = self.plan[0]
                if exp_act == self.prev_act and exp_key == key:
                    self.plan.popleft()
                else:
                    self.stats["plan_diverged"] += 1
                    self.plan.clear()
        self.prev_levels = levels
        self.prev_grid = grid

        # 2. life budget: reset is free, lethal action consumes step; preempt death with voluntary reset
        if self.life_budget is not None and self.life >= self.life_budget - 1 and not self.plan:
            return self._reset("life_budget_exhausted")

        # 2b. continue executing active BFS plan
        if self.plan:
            a, _ = self.plan[0]
            return self._emit(node, a, "plan")

        # 2c. directional goal guidance: track target and evaluate patience
        target = None
        if levels not in self.guide_off and any(a in avail for a in (1, 2, 3, 4)):
            target = self._target(levels, grid)
        if target is not None and node.mover is not None:
            if self._l1(node.mover, target) <= 2.0:
                self.target_wait[levels] = self.target_wait.get(levels, 0) + 1
                if self.target_wait[levels] > TARGET_PATIENCE:
                    self.target_idx[levels] = self.target_idx.get(levels, 0) + 1
                    self.target_wait[levels] = 0
                    self.stats["target_switched"] += 1
                    target = self._target(levels, grid)

        reg = None
        if levels not in self.guide_off and 6 in avail:
            reg = self._h2_regions(levels, grid)
            if reg is not None:
                mis = self._h2_mismatch(grid, reg)
                if mis < self.h2_best.get(levels, 1 << 30):
                    self.h2_best[levels] = mis
                    self.h2_stale[levels] = 0
                else:
                    self.h2_stale[levels] = self.h2_stale.get(levels, 0) + 1
                    if self.h2_stale[levels] > H2_PATIENCE:
                        self.guide_off.add(levels)
                        self.stats["h2_falsified"] += 1
                        reg = None

        # 3. select untried candidate at current priority; else BFS to frontier; else increment priority
        p = self.p_level.get(levels, 0)
        while True:
            un = node.untried(p, self.reactive)
            if un:
                lo = min(node.eff_tier(c, self.reactive) for c in un)
                pool = [c for c in un if node.eff_tier(c, self.reactive) == lo]
                if target is not None and node.mover is not None:
                    sc = {c: self._goal_score(c, node.mover, target) for c in pool}
                    hi = max(sc.values())
                    if hi > 0:
                        pool = [c for c in pool if sc[c] == hi]
                        self.stats["guided"] += 1
                elif reg is not None:
                    h0, w0 = node.shape
                    def _mis(c):
                        if c[0] != 6:
                            return False
                        gx, gy = (c[1] * w0 // 64, c[2] * h0 // 64) if (w0, h0) != (64, 64) else (c[1], c[2])
                        return self._h2_cell_mismatch(grid, reg, gx, gy)
                    pref = [c for c in pool if _mis(c)]
                    if pref:
                        pool = pref
                        self.stats["guided_h2"] += 1
                c = self.rng.choice(pool)
                self.p_level[levels] = p
                return self._emit(node, c, "local")
            path = self._bfs(node, p, target, reg)
            if path and self.life_budget is not None and len(path) > self.life_budget - self.life - 1:
                sk = self.start_key.get(levels)
                sn = self.nodes.get(sk) if sk is not None else None
                alternate_path = self._bfs(sn, p) if sn is not None and sn.key != node.key else None
                if alternate_path is not None and len(alternate_path) < self.life_budget - 1 and len(alternate_path) <= len(path):
                    return self._reset("plan_over_budget")
            if path:
                self.plan = deque(path)
                self.p_level[levels] = p
                a, _ = self.plan[0]
                self.stats["plan_new"] += 1
                return self._emit(node, a, "plan")
            if p < MAX_TIER:
                p += 1
                if p == TIER_RANDOM and 6 in avail:
                    self._add_random_clicks(node)
                continue
            break

        # 4. exhausted graph on this level: retry tested actions (possible hidden state) or trigger reset
        self.exhaust[levels] = self.exhaust.get(levels, 0) + 1
        self.stats["exhausted"] += 1
        if levels not in self.guide_off:
            self.guide_off.add(levels)
            self.stats["guidance_disabled"] += 1
        if self.exhaust[levels] % EXHAUST_RESET == 1 and self.reopened.get(levels, 0) < REOPEN_MAX:
            self.reopened[levels] = self.reopened.get(levels, 0) + 1
            for n in self.nodes.values():
                if n.level == levels:
                    n.tried = {c: r for c, r in n.tried.items() if r == DEATH}
            self.p_level[levels] = 0
            self.stats["reopened"] += 1
            un = node.untried(MAX_TIER, self.reactive)
            if un:
                return self._emit(node, self.rng.choice(un), "reopened")
        if self.exhaust[levels] % EXHAUST_RESET == 0 or self.reopened.get(levels, 0) >= REOPEN_MAX:
            return self._reset("exhausted")
        if 6 in avail and self._add_random_clicks(node):
            un = node.untried(MAX_TIER, self.reactive)
            return self._emit(node, self.rng.choice(un), "random")
        pool = [k for k in node.cands if k[0] in node.legal] or [(a,) for a in avail]
        return self._emit(node, self.rng.choice(pool), "retry")


In [ ]:
%%writefile /kaggle/working/hybrid_analyzer.py
"""Hybrid Analyzer: integrates graph explorer K into the Duck/TAAF harness (Tufa Labs, MIT).

Provides the analyzer adapter for HarnessSolver via `analyzer_factory(game, index)`:
- KPolicy: adapts graph_explorer.py (Agent subclass) to the TAAF harness interface:
  builds FrameData from raw observations, queries graph explorer K, and translates actions
  to harness step_env dictionaries. Also observes third-party model actions to maintain the state graph.
- HybridAnalyzer: manages policy arbitration between the base model and K:
  evaluates certificates, masks keys certified to have no effect from this state,
  injects informative guidance into prompts, and invokes K when stall, failure,
  repetition, or time pressure triggers fire.
- Robustness layer: wraps step_env and exceptions to prevent analyzer failures from terminating
  evaluation runs, falling back gracefully to the original model.
"""
from __future__ import annotations

import importlib.util
import os
import random
import sys
import time
import zlib
from pathlib import Path
from typing import Any, Callable

import numpy as np

import arcengine
from arcengine import FrameData, GameAction, GameState

# ---------------------------------------------------------------------------------------------
# Locate graph_explorer.py and ARC-AGI-3-Agents shim
# ---------------------------------------------------------------------------------------------
_HERE = Path(__file__).resolve().parent


def _candidates() -> list[Path]:
    """Resolved per KPolicy instance, not on import."""
    return ([Path(os.environ["EXPLORER_MODULE_PATH"])] if os.environ.get("EXPLORER_MODULE_PATH") else []) + [
        _HERE.parent / "agents" / "graph_explorer.py",
        _HERE / "graph_explorer.py",
        Path("/kaggle/working/graph_explorer.py"),
    ]


def _ensure_agents_shim() -> None:
    """Shim providing the minimal Agent base interface when ARC-AGI-3-Agents is not in sys.path."""
    try:
        import agents.agent  # noqa: F401
        return
    except Exception:
        pass
    import types

    pkg = types.ModuleType("agents")
    mod = types.ModuleType("agents.agent")

    class Agent:  # minimal Agent interface compatible with vendor agents/agent.py
        MAX_ACTIONS: int = 80
        action_counter: int = 0

        def __init__(self, card_id, game_id, agent_name, ROOT_URL, record, arc_env, tags=None) -> None:
            self.card_id = card_id
            self.game_id = game_id
            self.agent_name = agent_name
            self.ROOT_URL = ROOT_URL
            self.arc_env = arc_env
            self.tags = tags or []
            self.guid = ""
            self.frames = [FrameData(levels_completed=0)]

        @property
        def name(self) -> str:
            return f"{self.game_id}.{self.__class__.__name__.lower()}"

    mod.Agent = Agent
    pkg.agent = mod
    sys.modules["agents"] = pkg
    sys.modules["agents.agent"] = mod


def _import_graph_explorer():
    _ensure_agents_shim()
    cands = _candidates()
    for p in cands:
        if p.exists():
            spec = importlib.util.spec_from_file_location("graph_explorer", p)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            return mod
    raise RuntimeError("graph_explorer.py not found: " + ", ".join(str(p) for p in cands))


try:
    from inference.agent.tool_agent import AnalyzerTurnResult  # type: ignore
except Exception:  # standalone testing fallback replicating FrameView dataclass
    from dataclasses import dataclass

    @dataclass(frozen=True)
    class AnalyzerTurnResult:  # type: ignore[no-redef]
        step_executed: bool
        retryable_failure: bool = False
        reasoning: str = ""
        yielded_control: bool = False


_MODEL_NAME = {1: "UP", 2: "DOWN", 3: "LEFT", 4: "RIGHT", 5: "SPACE", 6: "MOUSE", 7: "UNDO", 0: "RESET"}
_ENGINE_ID = {"ACTION1": 1, "ACTION2": 2, "ACTION3": 3, "ACTION4": 4, "ACTION5": 5, "ACTION6": 6, "ACTION7": 7, "RESET": 0,
              "UP": 1, "DOWN": 2, "LEFT": 3, "RIGHT": 4, "SPACE": 5, "MOUSE": 6, "UNDO": 7}
# Unsupported actions returning None without spending turn are marked invalid (mark_invalid) to prevent repeated proposals.
_NOOP, _CHANGE, _PROGRESS, _DEATH = 0, 1, 2, 3      # mirrors graph_explorer transition outcomes
HINT_MAX_CHARS = 700                                 # maximum prompt hint character budget


def _fmt_act(c: tuple) -> str:
    return _MODEL_NAME.get(c[0], str(c[0])) + (f"(row={c[2]},col={c[1]})" if c[0] == 6 else "")


def _first_action(arguments: dict[str, Any]) -> tuple | None:
    """Tuple representing the first action from a step_env request dictionary."""
    d = arguments if arguments.get("action") else ((arguments.get("actions") or [None])[0] or {})
    aid = _ENGINE_ID.get(str(d.get("action", "")).strip().upper())
    if aid is None or aid == 0:
        return None
    if aid == 6:
        return (6, int(d.get("col", 0)), int(d.get("row", 0)))
    return (aid,)


def _cap_lines(lines: list[str], max_chars: int = HINT_MAX_CHARS) -> list[str]:
    out, used = [], 0
    for ln in lines:
        if used + len(ln) + 1 > max_chars:
            break
        out.append(ln)
        used += len(ln) + 1
    return out


def _payload_ok(payload: Any) -> bool:
    return isinstance(payload, dict) and bool(payload) and not payload.get("error") and "executed_count" in payload


def _legal_fallback(game: Any, rng: random.Random) -> dict[str, Any] | None:
    """Fallback action sampled from current legal actions when an exception occurs."""
    try:
        ids = [int(a) for a in game.current_state.available_actions if int(a) in (1, 2, 3, 4, 5, 6)]
    except Exception:
        ids = []
    if not ids:
        return None
    aid = rng.choice(ids)
    if aid == 6:
        return {"action": "MOUSE", "row": rng.randrange(64), "col": rng.randrange(64)}
    return {"action": _MODEL_NAME[aid]}


def _game_key(game: Any, default: Any = "game") -> str:
    return str(getattr(game, "env_name", None) or getattr(game, "game_id", None) or default).split("-")[0]


class _DummyEnv:
    """Minimal environment placeholder for Agent base class initialization."""
    observation_space = None


class KPolicy:
    """Policy adapter integrating graph explorer into the harness solver interface."""

    def __init__(self, game: Any, max_actions: int = 10 ** 9) -> None:
        mod = _import_graph_explorer()
        self.cls = mod.MyAgent
        game_id = str(getattr(game, "env_name", None) or getattr(game, "game_id", "game")).split("-")[0]
        self.agent = self.cls(card_id="hybrid", game_id=game_id, agent_name=f"k.{game_id}",
                              ROOT_URL="http://localhost", record=False, arc_env=_DummyEnv(), tags=["hybrid"])
        self.agent.MAX_ACTIONS = max_actions
        self.frames: list[FrameData] = [FrameData(levels_completed=0)]
        self.game = game
        self.last_proposal: tuple | None = None
        self.hidden_levels: set[int] = set()           # shared set of levels detected to have hidden state

    # ---- visual observation ----
    def frame_from_state(self, state: Any) -> FrameData:
        raw = state.raw
        frames = [np.asarray(f).tolist() for f in raw.frame] if raw.frame is not None and len(raw.frame) else []
        return FrameData(game_id=str(getattr(raw, "game_id", "") or ""), frame=frames, state=raw.state,
                         levels_completed=int(raw.levels_completed), win_levels=int(getattr(raw, "win_levels", 0) or 0),
                         available_actions=[int(a) for a in raw.available_actions])

    def current_frame(self) -> FrameData:
        return self.frame_from_state(self.game.current_state)

    # ---- observation of external actions ----
    def observe(self, engine_action: str, data: dict | None, frame_after: FrameData) -> None:
        """Informs explorer that an engine action was executed and observes resulting frame."""
        aid = _ENGINE_ID.get(str(engine_action).upper())
        if aid is None:
            return
        if aid == 0:
            # RESET (auto or from model): explorer clears step memory
            self.agent.prev_key = self.agent.prev_act = None
            self.agent.prev_grid = None
            self.agent.plan.clear()
            self.agent.hud_reset = True
            self.agent.life = 0
            return
        if self.agent.prev_key is None:
            return
        if aid == 6 and data:
            self.agent.prev_act = (6, int(data.get("x", 0)), int(data.get("y", 0)))
        else:
            self.agent.prev_act = (aid,)

    # ---- policy proposal ----
    def propose(self, frame: FrameData | None = None) -> tuple[dict[str, Any] | None, dict[str, Any]]:
        """Returns step_env arguments or None for reset, accompanied by diagnostic reason."""
        frame = frame or self.current_frame()
        self.frames.append(frame)
        if len(self.frames) > 4:
            del self.frames[:-4]
        act = self.agent.choose_action(self.frames, frame)
        reasoning = getattr(act, "reasoning", None)
        if act is GameAction.RESET or act.name == "RESET":
            self.last_proposal = (0,)
            return {"action": "RESET"}, {"k": "reset", "why": reasoning}
        aid = int(act.value[0]) if isinstance(act.value, tuple) else int(act.value)
        if aid == 6:
            # Click coordinates are retrieved from instance prev_act to prevent cross-thread interference.
            pa = self.agent.prev_act
            if isinstance(pa, tuple) and len(pa) == 3 and pa[0] == 6:
                x, y = int(pa[1]), int(pa[2])
            else:
                d = getattr(self.agent, "_click", None) or (act.action_data.model_dump() if hasattr(act, "action_data") else {})
                x, y = int(d.get("x", 0)), int(d.get("y", 0))
            x, y = max(0, min(63, x)), max(0, min(63, y))
            self.last_proposal = (6, x, y)
            return {"action": "MOUSE", "row": y, "col": x}, {"k": reasoning}
        self.last_proposal = (aid,)
        return {"action": _MODEL_NAME.get(aid, f"ACTION{aid}")}, {"k": reasoning}

    def absorb(self, frame: FrameData) -> None:
        """Processes observed frame into state graph without executing explorer proposal. Level transition resets life to zero."""
        ag = self.agent
        life, max_life, hud, steps, lv0 = ag.life, ag.max_life, ag.hud_reset, ag.steps, ag.prev_levels
        args, _ = self.propose(frame)
        new_level = int(frame.levels_completed) > lv0
        if args.get("action") != "RESET" or frame.state in (GameState.GAME_OVER, GameState.NOT_PLAYED):
            if new_level:
                ag.life = 0
            return
        grid = frame.frame[-1] if frame.frame else []
        if not grid:
            return
        ag.prev_grid = grid
        ag.prev_key = ag._key_of(grid, int(frame.levels_completed))
        ag.prev_act = None
        ag.life = 0 if new_level else life + 1
        ag.max_life = max(max_life, ag.life)
        ag.hud_reset = hud if ag.steps == steps else False
        self.last_proposal = None

    def mark_invalid(self) -> None:
        """Marks last proposed action as invalid/tested when rejected by environment."""
        ag = self.agent
        node = ag.nodes.get(ag.prev_key) if ag.prev_key is not None else None
        if node is not None and self.last_proposal and self.last_proposal[0] != 0:
            node.tried[self.last_proposal] = 0
        ag.prev_key = ag.prev_act = None

    # ---- cross-level persistence ----
    def cross_level_lines(self, max_items: int = 6) -> list[str]:
        """Returns persistent findings across level transitions."""
        ag = self.agent
        lines = ["Cross-level notes from the explorer (persist across levels; informational only):"]
        react = sorted(((ag.cls_tries[k], k) for k in ag.reactive), reverse=True)[:max_items]
        if react:
            lines.append("- component classes that REACTED to a click earlier in this game (color,size,w,h; times tried): "
                         + ", ".join(f"{k}×{n}" for n, k in react))
        dead = sorted(((n, k) for k, n in ag.cls_tries.items() if k not in ag.reactive and n >= 3), reverse=True)[:max_items]
        if dead:
            lines.append("- component classes clicked ≥3 times with NO effect so far: " + ", ".join(f"{k}×{n}" for n, k in dead))
        dirs = []
        for a in (1, 2, 3, 4):
            d = self.agent._dir_of(a) if hasattr(self.agent, "_dir_of") else None
            if d:
                dirs.append(f"{_MODEL_NAME[a]} moves the mover by {d}")
        if dirs:
            lines.append("- learned key effects (dx,dy sign): " + "; ".join(dirs))
        if ag.life_budget:
            lines.append(f"- a life/energy budget of about {ag.life_budget} actions per attempt was detected (game over when exhausted)")
        per_level = {}
        for n in ag.nodes.values():
            per_level[n.level] = per_level.get(n.level, 0) + 1
        if per_level:
            lines.append("- distinct states explored per level so far: " + ", ".join(f"L{l + 1}:{c}" for l, c in sorted(per_level.items())))
        return lines if len(lines) > 1 else []

    # ---- prompt guidance ----
    def hint_lines(self, max_items: int = 8) -> list[str]:
        ag = self.agent
        key = ag.prev_key
        node = ag.nodes.get(key) if key is not None else None
        lines = ["Explorer notes (graph-based, deterministic; informational only):"]
        if node is None:
            lines.append("- no graph knowledge for this exact state yet")
            return lines
        noop = [c for c, r in node.tried.items() if r == 0]
        untried = [c for c in node.cands if c not in node.tried]
        fmt = lambda c: _MODEL_NAME[c[0]] + (f"(row={c[2]},col={c[1]})" if c[0] == 6 else "")
        if noop and node.level in self.hidden_levels:   # hidden state levels cannot certify non-effects
            lines.append(f"- actions that changed nothing here before, but this level has hidden state, so that is NOT a proof: "
                         f"{', '.join(fmt(c) for c in noop[:max_items])}")
        elif noop:
            lines.append(f"- actions already proven to change NOTHING from this exact state: {', '.join(fmt(c) for c in noop[:max_items])}")
        if untried:
            lines.append(f"- actions not yet tried from this exact state: {', '.join(fmt(c) for c in untried[:max_items])}" + (" ..." if len(untried) > max_items else ""))
        lvl = ag.prev_levels
        n_states = sum(1 for n in ag.nodes.values() if n.level == lvl)
        lines.append(f"- distinct states seen on this level: {n_states}; actions this life: {ag.life}" + (f" of a budget of {ag.life_budget}" if ag.life_budget else ""))
        return lines


# ---------------------------------------------------------------------------------------------
# Autonomous Graph Explorer Mode
# ---------------------------------------------------------------------------------------------
class KAnalyzer:
    label = "k-analyzer"

    def __init__(self, game: Any, max_actions: int = 10 ** 9) -> None:
        self.policy = KPolicy(game, max_actions=max_actions)
        self.game = game
        self.total_tokens = 0          # harness tracks model tokens; explorer consumes zero LLM tokens
        self.steps = 0
        self.stats: dict[str, int] = {"layer_exc": 0, "k_exc": 0}
        self.last_exc = ""
        self._rng = random.Random(zlib.crc32(_game_key(game).encode()))

    def analyze(self, state_path: Path, action_num: int, valid_actions=None, step_env: Callable | None = None,
                transcript_path: Path | None = None, analysis_step: int | None = None, transcript_updated=None,
                request_timeout_seconds=None, should_stop: Callable[[], bool] | None = None, **_: Any):
        if step_env is None:
            return AnalyzerTurnResult(step_executed=False, reasoning="no step_env")
        if should_stop is not None and should_stop():
            return AnalyzerTurnResult(step_executed=False, reasoning="stopped")
        try:                                             # errors trigger legal fallback action, logged and handled gracefully
            args, diag = self.policy.propose()
        except Exception as e:
            self.stats["k_exc"] += 1
            self.last_exc = f"{type(e).__name__}: {e}"[:200]
            try:
                self.policy.agent.prev_key = self.policy.agent.prev_act = None
            except Exception:
                pass
            self.policy.last_proposal = None
            args, diag = _legal_fallback(self.game, self._rng), {"k": "exc"}
            if args is None:
                return AnalyzerTurnResult(step_executed=False, retryable_failure=True, reasoning="k_exc without legal action")
        payload = step_env(args)
        self.steps += 1
        ok = _payload_ok(payload)
        if transcript_path is not None:
            try:
                with open(transcript_path, "a", encoding="utf-8") as f:
                    f.write(f"[k-analyzer step {analysis_step}] {args} -> ok={ok} {str(diag)[:120]}\n")
            except Exception:
                pass
        if not ok:
            # invalid action rejected by environment: mark tested and yield without consuming step
            try:
                self.policy.mark_invalid()
            except Exception as e:
                self.stats["layer_exc"] += 1
                self.last_exc = f"{type(e).__name__}: {e}"[:200]
        return AnalyzerTurnResult(step_executed=ok, reasoning=str(diag)[:200])


# ---------------------------------------------------------------------------------------------
# Hybrid Analyzer: model by default, explorer takes over on failure, stall, repetition or timeout
# ---------------------------------------------------------------------------------------------
class HybridAnalyzer:
    label = "hybrid-duck-k"

    def __init__(self, game: Any, tool_agent: Any, *, stall_steps: int = 3, repeat_noop_limit: int = 2,
                 time_floor_s: float = 600.0, level_share: float = 0.6, max_runtime_s: float | None = None,
                 inject_hint: bool = True, k_burst: int = 1, selector: str = "triggers", probe_actions: int = 20,
                 yield_window: int = 40, cross_level: bool = False, cross_level_turns: int = 5,
                 h8: str | None = None, refuse_limit: int = 2) -> None:
        # selector: "triggers" (fixed stall/fail triggers) | "yield" (yield-based arbitration per level)
        # h8: None (accounting only) | "list" (action masking + predicted outcome) | "refusal" (masking + limited refusal)
        assert h8 in (None, "list", "refusal"), h8
        self.h8 = h8
        self.refuse_limit = refuse_limit
        self.certs: dict[tuple, int] = {}            # (node_key, action) -> count of observed self-loop transitions
        self.refusals: dict[tuple, int] = {}
        self.hidden_levels: set[int] = set()         # levels where a certified non-effect was falsified
        self._reindex_seen = 0
        self._reopened_seen: dict[int, int] = {}
        self._last_valid: list[str] = []
        self.max_hint_chars = 0
        self.selector = selector
        self.probe_actions = probe_actions
        self.yield_window = yield_window
        self.cross_level = cross_level
        self.cross_level_turns = cross_level_turns
        self.outcomes: dict[str, list[tuple[int, int]]] = {"model": [], "k": []}   # (effective?, progress?) per action
        self.turns_in_level = 0
        self.level_of_turns = 0
        self.game = game
        self.model = tool_agent
        self.policy = KPolicy(game)
        self.policy.hidden_levels = self.hidden_levels   # shared reference to hidden state levels
        self._hidden_death_seen = 0                   # hidden deaths already processed
        self.stall_steps = stall_steps
        self.repeat_noop_limit = repeat_noop_limit
        self.time_floor_s = time_floor_s
        self.level_share = level_share
        self.max_runtime_s = max_runtime_s
        self.inject_hint = inject_hint
        self.k_burst = k_burst
        self.t0 = time.monotonic()
        self.level_t0 = time.monotonic()
        self.level_seen = 0
        self.no_step = 0
        self.noop_repeats = 0
        self.k_turns = 0
        self.model_turns = 0
        self.stats: dict[str, int] = {"k_failure": 0, "k_stall": 0, "k_repeat": 0, "k_time": 0, "k_yield": 0, "model_yield": 0,
                                      # non-effect certification tracker (always active)
                                      "cert_emitted": 0, "cert_falsified": 0, "cert_hidden_death": 0, "cert_revoked_reindex": 0, "cert_revoked_reopen": 0,
                                      "model_act_total": 0, "model_act_certified": 0, "model_batches": 0, "model_deaths": 0,
                                      "h8_masked_keys": 0, "h8_refused": 0,
                                      # Layer and model exceptions counted and tracked
                                      "layer_exc": 0, "k_exc": 0, "model_exc": 0, "fallback_act": 0}
        self.last_exc = ""
        self._rng = random.Random(zlib.crc32(_game_key(game).encode()))
        self._executed = 0                               # actions executed via step_env on this instance
        self._patch_prompt()

    def _note_exc(self, key: str, e: BaseException) -> None:
        self.stats[key] = self.stats.get(key, 0) + 1
        self.last_exc = f"{key}: {type(e).__name__}: {e}"[:200]

    # Forward solver timeout and token count attributes
    @property
    def _timeout(self):
        return getattr(self.model, "_timeout", None)

    @property
    def generated_tokens(self) -> int:
        try:
            m = self.model
            v = getattr(m, "generated_tokens", None) if hasattr(m, "generated_tokens") else getattr(m, "total_tokens", 0)
            return max(0, int(v or 0))
        except Exception:
            return 0

    # ---- Non-effect certificates ----
    def _drop_level(self, level: int) -> int:
        nodes = self.policy.agent.nodes
        gone = [c for c in self.certs if (nodes.get(c[0]) is None or nodes[c[0]].level == level)]
        for c in gone:
            self.certs.pop(c, None)
            self.refusals.pop(c, None)
        return len(gone)

    def _cert_sync(self, before_key, before_act, batch_n: int = 1, died: bool = False) -> None:
        """Updates non-effect certificates after frame processing: emits, confirms, falsifies, or revokes."""
        ag = self.policy.agent
        ri = int(ag.stats.get("reindex", 0))
        if ri != self._reindex_seen:                     # HUD mask changed: node keys reindexed, revoke obsolete certificates
            self._reindex_seen = ri
            self.stats["cert_revoked_reindex"] += len(self.certs)
            self.certs.clear()
            self.refusals.clear()
        for lvl, n in list(ag.reopened.items()):         # explorer reopened tested actions: potential hidden state, revoke certificates
            if n != self._reopened_seen.get(lvl):
                self._reopened_seen[lvl] = n
                self.stats["cert_revoked_reopen"] += self._drop_level(lvl)
        mo = int(ag.stats.get("hidden_death", 0))
        if mo != self._hidden_death_seen:
            # Death on previously safe edge indicates hidden state: revoke level certificates and mark hidden
            self._hidden_death_seen = mo
            nb = ag.nodes.get(before_key) if before_key is not None else None
            lvl = nb.level if nb is not None else int(ag.prev_levels)
            self.stats["cert_hidden_death"] += 1
            self.hidden_levels.add(lvl)
            self._drop_level(lvl)
        if before_key is None or before_act is None or batch_n != 1:
            return                                       # batch execution: edge does not correspond to single action, skip certification
        node = ag.nodes.get(before_key)
        if node is None:
            return
        r = node.tried.get(before_act)
        c = (before_key, before_act)
        if c in self.certs or (died and r == _NOOP):
            # Lethal outcomes invalidate certificates; unexpected death indicates hidden state.
            if r in (_CHANGE, _PROGRESS, _DEATH) or died:  # falsified certificate implies hidden state exists in current level
                self.stats["cert_falsified"] += 1
                self.hidden_levels.add(node.level)
                self._drop_level(node.level)
            elif r == _NOOP:
                self.certs[c] += 1
        elif r == _NOOP and node.level not in self.hidden_levels:  # lethal outcome handled above: never certify non-effect on GAME_OVER
            self.certs[c] = 1
            self.stats["cert_emitted"] += 1

    def _masked_here(self) -> tuple[set[str], list[tuple]]:
        key = self.policy.agent.prev_key
        names: set[str] = set()
        clicks: list[tuple] = []
        if key is None:
            return names, clicks
        for (k, a) in self.certs:
            if k != key:
                continue
            if a[0] == 6:
                clicks.append(a)
            else:
                names.add(_MODEL_NAME[a[0]])
        return names, clicks

    def _filter_valid(self, valid_actions) -> list[str] | None:
        """Filters certified non-effect keys from advisory action list (preserves MOUSE and UNDO)."""
        if self.h8 is None or not isinstance(valid_actions, list):
            return valid_actions
        names, _ = self._masked_here()
        if not names:
            return valid_actions
        keep = self._mask(valid_actions, names)
        if keep is None:
            return valid_actions
        self.stats["h8_masked_keys"] += len(valid_actions) - len(keep)
        return keep

    @staticmethod
    def _mask(valid_actions: list, names: set[str]) -> list | None:
        """Returns masked action list; returns None if filtering would yield an empty list."""
        keep = [v for v in valid_actions if _MODEL_NAME.get(_ENGINE_ID.get(str(v).strip().upper(), -1), None) not in names]
        return keep if keep else None

    def h8_lines(self, max_items: int = 6) -> list[str]:
        """Formats prompt section detailing masked non-effect actions and predicted outcomes."""
        ag = self.policy.agent
        key = ag.prev_key
        node = ag.nodes.get(key) if key is not None else None
        if node is None:
            return []
        lines: list[str] = []
        names, clicks = self._masked_here()
        if names:
            # Report only actions actually removed; if all actions were masked and restored, do not announce removal
            valid = list(self._last_valid)
            keep = self._mask(valid, names) if valid else []
            if valid and keep is None:
                lines.append("- every key in your valid list was verified to change nothing from this exact state (list kept "
                             "whole): " + ", ".join(sorted(names)) + "; prefer MOUSE, RESET or a different approach")
            else:
                shown = {_MODEL_NAME.get(_ENGINE_ID.get(str(v).strip().upper(), -1)) for v in valid} if valid else set(names)
                removed = sorted(n for n in names if n in shown)
                if removed:
                    lines.append("- removed from your valid list (verified to change nothing from this exact state): " + ", ".join(removed))
        if clicks:
            lines.append("- clicks verified to change nothing here: " + ", ".join(_fmt_act(c) for c in clicks[:max_items]) + (" ..." if len(clicks) > max_items else ""))
        preds = []
        for a, t in list(node.edges.items())[:max_items * 2]:
            r = node.tried.get(a)
            if r == _NOOP or t == node.key:
                continue
            tgt = ag.nodes.get(t)
            if r == _PROGRESS:
                preds.append(f"{_fmt_act(a)} → COMPLETED THE LEVEL before")
            elif tgt is not None:
                n_un = len([c for c in tgt.cands if c not in tgt.tried])
                preds.append(f"{_fmt_act(a)} → state seen before ({n_un} untried actions there)")
        deaths = [a for a, r in node.tried.items() if r == _DEATH]
        for a in deaths[:2]:
            preds.append(f"{_fmt_act(a)} → GAME OVER before" + (f" (life budget ≈ {ag.life_budget})" if ag.life_budget else ""))
        if preds:
            lines.append("- predicted outcomes of known actions here: " + "; ".join(preds[:max_items]))
        if node.level in self.hidden_levels:
            lines.append("- caution: on this level the same screen evolved differently twice (hidden state); graph notes are weaker")
        return lines

    # ---- yield-based arbitration ----
    def _record(self, who: str, payload: dict[str, Any]) -> None:
        if not payload or payload.get("error") or "executed_count" not in payload:
            return
        n = int(payload.get("executed_count") or 1)
        eff = 1 if payload.get("board_changed") else 0
        prog = 1 if payload.get("level_completed") else 0
        lst = self.outcomes[who]
        lst.extend([(eff, prog)] * max(1, n))
        if len(lst) > 4 * self.yield_window:
            del lst[:-2 * self.yield_window]
        lvl = int(self.game.current_state.levels_completed)
        if lvl != self.level_of_turns:
            self.level_of_turns = lvl
            self.turns_in_level = 0
            self.outcomes = {"model": [], "k": []}            # yield measured per level

    def _yield(self, who: str) -> float | None:
        w = self.outcomes[who][-self.yield_window:]
        if len(w) < self.probe_actions:
            return None
        return sum(10 * p + e for e, p in w) / len(w)

    def _yield_says_k(self) -> bool:
        ym, yk = self._yield("model"), self._yield("k")
        if ym is None:
            return False                                      # model not yet probed on this level
        if yk is None:
            return True                                       # probe explorer on this level
        return yk > ym

    # inject prompt guidance
    def _patch_prompt(self) -> None:
        if not self.inject_hint or not hasattr(self.model, "_build_user_prompt"):
            return
        orig = self.model._build_user_prompt
        policy = self.policy

        hyb = self

        def patched(action_num, **kw):
            text = orig(action_num, **kw)
            try:
                lines = policy.hint_lines()
                if hyb.h8 is not None:
                    lines = lines + hyb.h8_lines()
                if hyb.cross_level and hyb.turns_in_level < hyb.cross_level_turns:
                    lines = policy.cross_level_lines() + lines
                lines = _cap_lines(lines)
                hyb.max_hint_chars = max(hyb.max_hint_chars, sum(len(l) + 1 for l in lines))
                parts = text.split("\n", 2)
                if len(parts) == 3:
                    return parts[0] + "\n" + parts[1] + "\n" + "\n".join(lines) + "\n" + parts[2]
                return text + "\n" + "\n".join(lines)
            except Exception as e:
                hyb._note_exc("layer_exc", e)
                return text
        self.model._build_user_prompt = patched

    # observe all executed actions
    def _wrap_step_env(self, step_env: Callable) -> Callable:
        policy = self.policy
        hyb = self

        def wrapped(arguments: dict[str, Any]) -> dict[str, Any]:
            # Guarded execution ensuring model request reaches step_env exactly once
            before_key, first, cert, had_cert = None, None, None, False
            try:
                before_key = policy.agent.prev_key
                first = _first_action(arguments)
                cert = (before_key, first) if (before_key is not None and first is not None) else None
                had_cert = cert in hyb.certs if cert else False
                # limited refusal without spending action; executes after refuse_limit attempts
                if hyb.h8 == "refusal" and had_cert and hyb.refusals.get(cert, 0) < hyb.refuse_limit:
                    hyb.refusals[cert] = hyb.refusals.get(cert, 0) + 1
                    hyb.stats["h8_refused"] += 1
                    return {"executed": False, "h8_refused": True,
                            "error": (f"{_fmt_act(first)} was NOT executed: the explorer verified {hyb.certs[cert]}x that it changes "
                                      "nothing from this exact state. Choose a different action."),
                            "valid_actions": hyb._filter_valid(list(hyb._last_valid))}
            except Exception as e:
                hyb._note_exc("layer_exc", e)
                before_key, first, cert, had_cert = None, None, None, False
            payload = step_env(arguments)
            if _payload_ok(payload):
                hyb._executed += 1                       # record execution before layer processing to prevent re-execution
            try:
                hyb._record("model", payload)
                executed = payload.get("executed_actions") or []
                if executed and not payload.get("error"):
                    n_exec = len(executed)
                    hyb.stats["model_act_total"] += 1
                    if had_cert:
                        hyb.stats["model_act_certified"] += 1
                    if n_exec > 1:
                        hyb.stats["model_batches"] += 1
                    # only the final action of a batch has visible resulting frame; record it
                    name = str(executed[-1]).split("(")[0].strip().upper()
                    eng_id = _ENGINE_ID.get(name)
                    data = None
                    if eng_id is not None:
                        if eng_id == 6:
                            # In batch execution, only the n_exec-th requested action corresponds to the final visible frame
                            _lst = [arguments] if arguments.get("action") else list(arguments.get("actions") or [{}])
                            d = _lst[min(n_exec, len(_lst)) - 1]
                            # Harness clamps coordinates to 0..63 before execution; explorer observes identical clamped click
                            data = {"x": max(0, min(63, int(d.get("col", 0)))), "y": max(0, min(63, int(d.get("row", 0))))}
                        policy.observe("RESET" if eng_id == 0 else f"ACTION{eng_id}", data, None)
                    if n_exec > 1:
                        # batch: edge (previous node, last action) is unobserved; update node without edge learning
                        policy.agent.prev_act = None
                    act_t = policy.agent.prev_act
                    frame = policy.current_frame()
                    was_noop = False
                    node_before = policy.agent.nodes.get(before_key) if before_key is not None else None
                    if frame.state is GameState.GAME_OVER:
                        hyb.stats["model_deaths"] += 1
                    elif node_before is not None and act_t is not None and node_before.tried.get(act_t) == 0:
                        was_noop = True
                    # Explorer processes new frame (edge, graph; on GAME_OVER records death). Explorer proposal is absorbed without mutating decision state.
                    policy.absorb(frame)
                    hyb._cert_sync(before_key, act_t, n_exec, died=frame.state is GameState.GAME_OVER)
                    if frame.state is not GameState.GAME_OVER:
                        # discarded proposal must not create spurious self-loop edges on subsequent takeover
                        policy.agent.prev_act = None
                        hyb.noop_repeats = hyb.noop_repeats + 1 if was_noop else 0
                    if isinstance(payload.get("valid_actions"), list):
                        payload["valid_actions"] = hyb._filter_valid(payload["valid_actions"])
            except Exception as e:
                hyb._note_exc("layer_exc", e)
            return payload
        return wrapped

    def _time_pressure(self) -> bool:
        if self.max_runtime_s is None:
            return False
        elapsed = time.monotonic() - self.t0
        remaining = self.max_runtime_s - elapsed
        if remaining < self.time_floor_s:
            return True
        lvl = int(self.game.current_state.levels_completed)
        if lvl != self.level_seen:
            self.level_seen = lvl
            self.level_t0 = time.monotonic()
        return (time.monotonic() - self.level_t0) > self.level_share * self.max_runtime_s

    def _fallback_turn(self, step_env: Callable, why: str) -> Any:
        """Fallback action when an unexpected error occurs; ensures run continuity."""
        try:
            ag = self.policy.agent
            ag.prev_key = ag.prev_act = None             # fallback action is not an explorer decision: do not record graph edge
        except Exception:
            pass
        self.policy.last_proposal = None
        args = _legal_fallback(self.game, self._rng)
        if args is None:
            return AnalyzerTurnResult(step_executed=False, retryable_failure=True, reasoning=f"{why}: without legal action")
        payload = step_env(args)
        ok = _payload_ok(payload)
        if ok:
            self._executed += 1
            self.stats["fallback_act"] += 1
        self.no_step = 0
        self.noop_repeats = 0
        return AnalyzerTurnResult(step_executed=ok, reasoning=f"fallback({why}) {args}")

    def _k_turn(self, step_env: Callable, why: str, transcript_path: Path | None, analysis_step):
        self.stats[why] += 1
        self.k_turns += 1
        try:
            args, diag = self.policy.propose()
        except Exception as e:                           # catch error to prevent evaluation crash
            self._note_exc("k_exc", e)
            return self._fallback_turn(step_env, why)
        ag = self.policy.agent
        k_key, k_act = ag.prev_key, ag.prev_act          # explorer action to be learned after execution
        payload = step_env(args)
        ok = _payload_ok(payload)
        if ok:
            self._executed += 1
        try:
            self._record("k", payload)
            if ok and k_act is not None:
                # Explorer absorbs frame from its own executed action to ensure correct edge and state attribution before subsequent model steps.
                life, max_life, prev_level = ag.life, ag.max_life, int(ag.prev_levels)
                k_frame = self.policy.current_frame()
                self.policy.absorb(k_frame)
                self._cert_sync(k_key, k_act, 1, died=k_frame.state is GameState.GAME_OVER)
                if ag.prev_key is not None:               # non-terminal frame: extra proposal is not an action
                    ag.life = 0 if int(ag.prev_levels) > prev_level else life  # level transition resets life budget in new level
                    ag.max_life = max_life
                    ag.prev_act = None
            if not ok:
                self.policy.mark_invalid()
        except Exception as e:
            self._note_exc("layer_exc", e)
        if transcript_path is not None:
            try:
                with open(transcript_path, "a", encoding="utf-8") as f:
                    f.write(f"[hybrid: K takes over ({why}) step {analysis_step}] {args} -> ok={ok}\n")
            except Exception:
                pass
        self.no_step = 0
        self.noop_repeats = 0
        return AnalyzerTurnResult(step_executed=ok, reasoning=f"K({why}) {str(diag)[:160]}")

    def analyze(self, state_path: Path, action_num: int, valid_actions=None, step_env: Callable | None = None,
                transcript_path: Path | None = None, analysis_step: int | None = None, transcript_updated=None,
                request_timeout_seconds=None, should_stop: Callable[[], bool] | None = None, **kw: Any):
        """Entry point invoked by harness solver. Handles errors gracefully to avoid run termination."""
        if step_env is None:
            return AnalyzerTurnResult(step_executed=False, reasoning="no step_env")
        n0 = self._executed
        try:
            return self._analyze(state_path, action_num, valid_actions, step_env, transcript_path, analysis_step,
                                 transcript_updated, request_timeout_seconds, should_stop, kw)
        except Exception as e:
            self._note_exc("layer_exc", e)
            if self._executed > n0:
                return AnalyzerTurnResult(step_executed=True, reasoning="layer_exc after action")
            try:
                return self._fallback_turn(step_env, "layer_exc")
            except Exception as e2:
                self._note_exc("layer_exc", e2)
                return AnalyzerTurnResult(step_executed=False, retryable_failure=True, reasoning="layer_exc")

    def _analyze(self, state_path, action_num, valid_actions, step_env, transcript_path, analysis_step,
                 transcript_updated, request_timeout_seconds, should_stop, kw):
        # Evaluate takeover triggers
        if self.no_step >= self.stall_steps:
            return self._k_turn(step_env, "k_stall", transcript_path, analysis_step)
        if self.noop_repeats >= self.repeat_noop_limit:
            return self._k_turn(step_env, "k_repeat", transcript_path, analysis_step)
        if self._time_pressure():
            return self._k_turn(step_env, "k_time", transcript_path, analysis_step)
        if self.selector == "yield" and self._yield_says_k():
            return self._k_turn(step_env, "k_yield", transcript_path, analysis_step)
        # Model turn, observed by explorer
        self.model_turns += 1
        self.turns_in_level += 1
        if self.selector == "yield":
            self.stats["model_yield"] += 1
        if isinstance(valid_actions, list):
            self._last_valid = list(valid_actions)
        valid_actions = self._filter_valid(valid_actions)
        n0 = self._executed
        try:
            result = self.model.analyze(state_path, action_num, valid_actions=valid_actions, step_env=self._wrap_step_env(step_env),
                                        transcript_path=transcript_path, analysis_step=analysis_step,
                                        transcript_updated=transcript_updated, request_timeout_seconds=request_timeout_seconds,
                                        should_stop=should_stop, **kw)
        except Exception as e:                           # Model ToolAgent failed: explorer assumes control instead of crashing
            self._note_exc("model_exc", e)
            if self._executed > n0:
                return AnalyzerTurnResult(step_executed=True, reasoning="model_exc after action")
            result = None
        if result is None or result.retryable_failure:
            # Model serving unavailable: explorer takes over to maintain progress
            return self._k_turn(step_env, "k_failure", transcript_path, analysis_step)
        if not result.step_executed and not getattr(result, "yielded_control", False):
            self.no_step += 1
        elif result.step_executed:
            self.no_step = 0
        return result

    @property
    def total_tokens(self) -> int:
        try:
            return int(getattr(self.model, "total_tokens", 0) or 0)
        except Exception:
            return 0


# ---------------------------------------------------------------------------------------------
# Factory for `bm.solver.analyzer_factory`
# ---------------------------------------------------------------------------------------------
def k_factory(max_actions: int = 10 ** 9):
    def make(game, index):
        a = KAnalyzer(game, max_actions=max_actions)
        REGISTRY[_game_key(game, index)] = a               # registry for final metrics inspection
        return a
    return make


REGISTRY: dict[str, Any] = {}      # game -> last analyzer created (for final diagnostic logging)
FACTORY_ERRORS: dict[str, str] = {}  # game -> error during construction (fallback to pure model)


def hybrid_factory(make_tool_agent: Callable[[Any, int], Any], **opts):
    """Factory constructing HybridAnalyzer instance with specified arbitration options."""
    def make(game, index):
        model = make_tool_agent(game, index)            # from harness: unexpected exception propagated as in original
        try:
            a = HybridAnalyzer(game, model, **opts)
        except Exception as e:
            # If explorer construction fails, proceed with base model
            FACTORY_ERRORS[_game_key(game, index)] = f"{type(e).__name__}: {e}"[:200]
            return model
        REGISTRY[_game_key(game, index)] = a
        return a
    return make


In [ ]:
# Hybrid Explorer Hook: model by default; graph explorer K assumes control on failure, stall, repetition, or time pressure;
# explorer graph provides informative advisory notes and observes all actions.
# Fallback guard: failures in this hook revert cleanly to the original harness ToolAgent.
import json, os, sys, traceback
EXPLORER_OPTS = dict(inject_hint=True, stall_steps=3, repeat_noop_limit=2, time_floor_s=600.0, level_share=0.6,
                      selector='triggers', probe_actions=20, yield_window=40, cross_level=False, cross_level_turns=5,
                      h8='list', refuse_limit=2)
try:
    sys.path.insert(0, "/kaggle/working")
    if SETUP_ENV_PATH.exists():
        os.environ.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    import hybrid_analyzer
    from inference.agent.tool_agent import ToolAgent
    _solver = bm.solver
    def _make_model(game, index):
        return ToolAgent(model=_solver.model, timeout=_solver.analyzer_timeout, save_request_logs=_solver.save_request_logs)
    _hybrid = hybrid_analyzer.hybrid_factory(_make_model, max_runtime_s=_solver.max_runtime_s_per_game, **EXPLORER_OPTS)
    def _explorer_factory(game, index):
        try:
            return _hybrid(game, index)
        except Exception as _e:
            print(f"[explorer] HybridAnalyzer failed on game {index} ({type(_e).__name__}: {_e}); using original ToolAgent", flush=True)
            return _make_model(game, index)
    bm.solver.analyzer_factory = _explorer_factory
    bm.label = f"{bm.label}-k-hybrid-h8-list"
    print(f"[explorer] analyzer_factory = HybridAnalyzer({EXPLORER_OPTS})")
except Exception as _e:
    bm.solver.analyzer_factory = None
    print(f"[explorer] explorer hook disabled, using original harness ({type(_e).__name__}: {_e})", flush=True)
    traceback.print_exc()


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and play the **live competition Arcade**. Otherwise, an interactive **Save & Run All** plays the fixed phase-A subset (`vc33`, `tu93`, `ls20`) offline, without gateway connectivity. Teardown commands run afterward even if the run raises.


In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)
    PHASE_A_GAMES = ['vc33', 'tu93', 'ls20']
    bm.games = [next((g for g in bm.games if str(getattr(g, 'env_name', '')).startswith(p)), None) for p in PHASE_A_GAMES]
    assert None not in bm.games, 'Phase A sample incomplete, missing: ' + repr([p for p, g in zip(PHASE_A_GAMES, bm.games) if g is None])

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    if not TRUE_SUBMISSION:
        # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)
finally:
    for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
        print(f"taaf.kaggle: teardown command: {command}", flush=True)
        subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())


In [ ]:
if TRUE_SUBMISSION:
    print("[explorer] metrics only available in interactive execution")
else:
  try:
    import re as _re
    _rows = []
    for _r in bm.game_runs:
        _gid = str(_r.game_id).split("-")[0]
        _lv = int(getattr(_r, "levels_completed", 0) or 0)
        _m = _re.search(r"tokens=(\d+)", str(_r.solver_note or ""))
        _tok = int(_m.group(1)) if _m else 0
        _rows.append((_gid, _lv, len(_r.history), _tok, float(_r.final_score or 0.0)))
    print(f"{'game':6} {'levels':>6} {'actions':>6} {'tokens':>9} {'tok/level':>10} {'actions/level':>11} {'score':>7}")
    for _gid, _lv, _ac, _tok, _sc in sorted(_rows):
        print(f"{_gid:6} {_lv:6} {_ac:6} {_tok:9} {(_tok/_lv if _lv else float('nan')):10.0f} {(_ac/_lv if _lv else float('nan')):11.1f} {_sc:7.3f}")
    _L = sum(r[1] for r in _rows); _T = sum(r[3] for r in _rows); _A = sum(r[2] for r in _rows)
    print(f"TOTAL levels={_L} actions={_A} tokens={_T} tokens/level={(_T/_L if _L else float('nan')):.0f} actions/level={(_A/_L if _L else float('nan')):.1f} mean score={sum(r[4] for r in _rows)/max(1,len(_rows)):.4f}")
    try:
        import hybrid_analyzer as _ka
        _keys = ["model_act_total", "model_act_certified", "cert_emitted", "cert_falsified", "cert_revoked_reindex", "cert_revoked_reopen",
                 "h8_masked_keys", "h8_refused", "k_failure", "k_stall", "k_repeat", "k_time", "k_yield", "model_batches", "model_deaths",
                 "layer_exc", "k_exc", "model_exc", "fallback_act"]
        if _ka.REGISTRY:
            print("game   " + " ".join(f"{k[:10]:>10}" for k in _keys) + "  turns(model/K)  hidden  hint_max")
            for _gid, _a in sorted(_ka.REGISTRY.items()):
                print(f"{_gid:6} " + " ".join(f"{_a.stats.get(k, 0):10d}" for k in _keys)
                      + f"  {_a.model_turns}/{_a.k_turns}  {sorted(_a.hidden_levels)}  {_a.max_hint_chars}")
        for _gid, _err in sorted(getattr(_ka, "FACTORY_ERRORS", {}).items()):
            print(f"K disabled on {_gid}: {_err}")
    except Exception as _e:
        print("H8 stats unavailable:", _e)
  except Exception as _e:
    print("[explorer] metrics unavailable:", type(_e).__name__, _e)


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`. It is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")